# Step-by-step Sentinel-2 extraction 

This notebook runs the full AEREO pipeline — **Search**, **Prepare** and **Extract** — step by step. 
We query a STAC catalog for Sentinel-2 assets, build extraction tasks for the AOI grid, and run each task through read → reproject → write.

Each of these functions can be replaced by ANY function that conforms to the inputs and outputs of each step as follows:

| Step | Input | Output |
|------|-------|--------|
| Search | `(collections, intersects, start_datetime, end_datetime, **kwargs)` | `GeoDataFrame[AssetSchema]` |
| Build | `(GeoDataFrame[AssetSchema], ExtractionJob)` | `Sequence[ExtractionTask]` |
| Read | `ExtractionTask` | `xr.Dataset` |
| Preprocess (optional) | `xr.Dataset` | `xr.Dataset` |
| Reproject | `(xr.Dataset, ExtractionTask, **kwargs)` | `dict[str, xr.Dataset]` |
| Postprocess (optional) | `xr.Dataset` | `xr.Dataset` |
| Write | `(xr.Dataset, ExtractionTask, ExtractionPatch)` | `GeoDataFrame[ArtifactSchema]` |

# Configuring AOI, Grid, Patch and Extraction pipeline
Before we start, we must configure our `ExtractionJob`. 
This means defining the `grid` to use, the `patch` resolution we want (e.g. 10 m, 100 m), and the functions to use in the `extraction pipeline` — always conforming to the required inputs and outputs.

If we know all of this beforehand, we can run our pipeline anywhere by serializing the job, and define different pipelines using only config files!

![AEREO config interaction diagram](assets/aereo_configs.svg)


In [ ]:
from aereo.builtins.search import search_stac
from datetime import datetime, timezone
from shapely.geometry import Polygon

from aereo.pipeline import ExtractionJob
from aereo.interfaces import ExtractConfig, GridConfig, PatchConfig
from aereo.builtins import read_odc_stac, reproject_odc, write_geotiff

# AOI polygon — Chocón reservoir, Argentina (inlined so the notebook has no file dependencies).
aoi_polygon = Polygon(
    [
        (-68.90986824592407, -39.23705421799603),
        (-68.65925870907353, -39.23705421799603),
        (-68.65925870907353, -39.41589522092947),
        (-68.90986824592407, -39.41589522092947),
        (-68.90986824592407, -39.23705421799603),
    ]
)

grid_config = GridConfig(
    target_grid_dist=10_000,
)

patch_config = PatchConfig(
    resolution=10.0,
    margin=10.0,
)

extract_config = ExtractConfig(
    read=read_odc_stac,
    preprocess=[],
    reproject=reproject_odc,
    postprocess=[],
    write=write_geotiff,
)

job = ExtractionJob(
    name="sentinel2_sample",
    grid_config=grid_config,
    patch_config=patch_config,
    output_uri="/tmp/aereo_extraction",
    extract=extract_config,
    target_aoi=aoi_polygon,
)

## Step 1 — Search: `search_stac`

The search provider queries the STAC API and returns a GeoDataFrame of matched assets. Each row corresponds to one requested asset (e.g. `red`, `nir`) from one STAC item. All parameters are passed explicitly.


In [ ]:
assets = search_stac(
    stac_api_url="https://earth-search.aws.element84.com/v1",
    collections={"sentinel-2-l2a": ["red", "nir"]},
    intersects=aoi_polygon,
    start_datetime=datetime(2024, 1, 1, tzinfo=timezone.utc),
    end_datetime=datetime(2024, 1, 10, tzinfo=timezone.utc),
)

print(f"\u2713 Found {len(assets)} asset rows")
print("Columns:", list(assets.columns))
print("First rows:")
assets[["id", "collection", "channel_id", "crs", "href"]].head()

## Step 2 — Build tasks: `build_grouped_tasks`

The task builder takes the search-result GeoDataFrame and the `ExtractionJob` and produces a list of `ExtractionTask` objects. Tasks are grouped by start time and native CRS, and patches are chunked by `cells_per_task`.


In [ ]:
from aereo.builtins.task_builder import build_grouped_tasks

tasks = build_grouped_tasks(
    search_results=assets,
    job=job,
    cells_per_task=4,
)

print(f"\u2713 Built {len(tasks)} extraction task(s)")
for i, task in enumerate(tasks):
    print(f"Task {i}: {task}")

In [ ]:
# Inspect the first task in detail
task = tasks[0]

print("Task context:", dict(task.task_context))
print(f"Number of asset rows in task: {len(task.assets)}")
print(f"Number of patches in task: {len(task.patches)}")

patch = task.patches[0]
print(f"First patch id: {patch.id}")
print(f"Patch resolution: {patch.resolution} m")
print(f"Patch UTM CRS: {patch.utm_crs}")
print(f"Patch geobox shape: {patch.geobox.shape}")

# Extraction stages available on the task (delegated from job.extract)
print("Extraction stages:")
print("  read:", task.extract.read.__name__)
print(
    "  preprocess:", [getattr(p, "__name__", str(p)) for p in task.extract.preprocess]
)
print("  reproject:", task.extract.reproject.__name__)
print(
    "  postprocess:", [getattr(p, "__name__", str(p)) for p in task.extract.postprocess]
)
print("  write:", task.extract.write.__name__)

## Step 3 — Read: `read_odc_stac`

The reader reconstructs `pystac.Item` objects from the `stac_item` column and uses `odc.stac.load` to build a lazy `xarray.Dataset` in the native CRS of the STAC items.


In [ ]:
from aereo.builtins.read import read_odc_stac

# Call the reader directly with the task.
ds_native = read_odc_stac(task)

print("Native dataset:")
ds_native["red"].plot()

## Step 4 — Reproject: `reproject_odc`

The reprojector warps the native CRS dataset onto each patch's UTM GeoBox. It returns a mapping from `patch.id` to a reprojected `xr.Dataset`.


In [ ]:
from aereo.builtins.reproject import reproject_odc

# Call the reprojector directly with explicit parameters.
ds_per_patch = reproject_odc(
    ds=ds_native,
    task=task,
    resampling="nearest",
)

print(f"\u2713 Reprojected dataset available for {len(ds_per_patch)} patch(es)")
print("Patch IDs:", list(ds_per_patch.keys()))

## Step 5 — Write: `write_geotiff`

The writer serialises a patch's dataset to a GeoTIFF and returns a GeoDataFrame of written artifacts conforming to `ArtifactSchema`.


In [ ]:
from aereo.builtins.write import write_geotiff

# Write a single patch
patch = task.patches[0]
patch_artifacts = write_geotiff(
    ds=ds_per_patch[patch.id],
    task=task,
    patch=patch,
)

print(f"\u2713 Wrote {len(patch_artifacts)} artifact row(s)")
print("Columns:", list(patch_artifacts.columns))
print("Artifact rows:")
patch_artifacts[["id", "uri", "grid_cell", "start_time", "end_time"]]

In [ ]:
import rioxarray

rioxarray.open_rasterio(patch_artifacts.iloc[0].uri)[0].plot()

## Putting it together: `run_task`

`run_task(task)` executes the read → preprocess → reproject → postprocess → write loop for a single `ExtractionTask`. It is what `LocalExecutor` uses under the hood.


In [ ]:
from aereo.execution import run_task

artifacts_gdf = run_task(task)

print(f"\u2713 Single task produced {len(artifacts_gdf)} artifact row(s)")
artifacts_gdf[["id", "uri", "grid_cell", "geometry"]].head()

In [ ]:
rioxarray.open_rasterio(artifacts_gdf.iloc[3].uri)[0].plot()

## Batch execution with an executor

To run many tasks in parallel, pass the task list to an `Executor`. For COG/I/O-bound extractors like this one, `LocalExecutor(use_threads=True)` is the safer choice in Jupyter and avoids the pickling issues that can make `ProcessPoolExecutor` hang.


In [ ]:
from aereo.executors import LocalExecutor

executor = LocalExecutor(workers=4, use_threads=True)
artifacts = executor(tasks)

print(f"\u2713 Extracted {len(artifacts)} artifact row(s) from {len(tasks)} task(s)")
artifacts[["id", "uri", "grid_cell", "start_time"]].head()

In [ ]:
from aereo.viz import plot_artifact_patches

plot_artifact_patches(artifacts, ds_factor=10, cmap="viridis")